# LAB | Feature Engineering

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In [35]:
# Step 0 - Import libraries and dataset
# -------------------------------------

# Import the libraries needed for this section
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# Standard seed used throughout the notebook, for reproducibility
SEED = 1

# Load the Iris dataset as a DataFrame
iris_data = load_iris(as_frame=True)
iris_df = iris_data.data.copy()
iris_df['species'] = iris_data.target.map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})
iris_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   sepal length (cm)  150 non-null    float64
 1   sepal width (cm)   150 non-null    float64
 2   petal length (cm)  150 non-null    float64
 3   petal width (cm)   150 non-null    float64
 4   species            150 non-null    object 
dtypes: float64(4), object(1)
memory usage: 6.0+ KB


In [36]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


**Check the shape of your data**

In [37]:
spaceship.shape

(8693, 14)

**Check for data types**

In [38]:
spaceship.dtypes

PassengerId      object
HomePlanet       object
CryoSleep        object
Cabin            object
Destination      object
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name             object
Transported        bool
dtype: object

**Check for missing values**

In [39]:
spaceship.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

There are multiple strategies to handle missing data

- Removing all rows or all columns containing missing data.
- Filling all missing values with a value (mean in continouos or mode in categorical for example).
- Filling all missing values with an algorithm.

For this exercise, because we have such low amount of null values, we will drop rows containing any missing value. 

In [40]:
spaceship = spaceship.dropna()

- **Cabin** is too granular - transform it in order to obtain {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'T'}

In [41]:
spaceship["Cabin"].unique()

array(['B/0/P', 'F/0/S', 'A/0/S', ..., 'G/1499/S', 'G/1500/S', 'E/608/S'],
      shape=(5305,), dtype=object)

In [42]:
spaceship["Cabin"] = spaceship["Cabin"].str[0]

In [43]:
spaceship["Cabin"].unique()

array(['B', 'F', 'A', 'G', 'E', 'C', 'D', 'T'], dtype=object)

- Drop PassengerId and Name

In [44]:
spaceship = spaceship.drop(columns = ["PassengerId", "Name"])

- For non-numerical columns, do dummies.

In [45]:
spaceship_encoded = pd.get_dummies(
    spaceship,
    columns=["HomePlanet", "CryoSleep", "Cabin", "Destination", "VIP","RoomService"],
    drop_first=True
).astype(int)
spaceship_encoded.head()

,Age,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,...,RoomService_6103.0,RoomService_6256.0,RoomService_6899.0,RoomService_7172.0,RoomService_7406.0,RoomService_8142.0,RoomService_8151.0,RoomService_8168.0,RoomService_8586.0,RoomService_9920.0
0,39,0,0,0,0,0,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,24,9,25,549,44,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,58,3576,0,6715,49,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,33,1283,371,3329,193,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,16,70,151,565,2,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


**Perform Train Test Split**

In [46]:
X = spaceship[['HomePlanet', 'CryoSleep','Cabin','Destination', 'Age','VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']]
y = spaceship['Transported']

X_sample = X.sample(5, random_state=SEED)
y_sample = y.sample(5, random_state=SEED)

print('X sample:')
print(X_sample)
print()
print('y sample:')
print(y_sample)

X sample:
     HomePlanet CryoSleep Cabin    Destination   Age    VIP  RoomService  \
5606      Earth     False     G    TRAPPIST-1e   0.0  False          0.0   
8226      Earth     False     F  PSO J318.5-22  20.0  False          0.0   
1288      Earth      True     G    TRAPPIST-1e  27.0  False          0.0   
3220      Earth      True     G    TRAPPIST-1e  64.0  False          0.0   
4596      Earth     False     F    TRAPPIST-1e  18.0  False          0.0   

      FoodCourt  ShoppingMall     Spa  VRDeck  
5606        0.0           0.0     0.0     0.0  
8226      571.0          82.0     0.0     0.0  
1288        0.0           0.0     0.0     0.0  
3220        0.0           0.0     0.0     0.0  
4596        0.0          31.0  2121.0     0.0  

y sample:
5606    False
8226    False
1288    False
3220     True
4596    False
Name: Transported, dtype: bool


In [47]:
X_encoded = pd.get_dummies(
    spaceship,
    columns=["HomePlanet", "CryoSleep", "Cabin", "Destination", "VIP","RoomService"],
    drop_first=True
).astype(int)
spaceship_encoded.head()

,Age,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,...,RoomService_6103.0,RoomService_6256.0,RoomService_6899.0,RoomService_7172.0,RoomService_7406.0,RoomService_8142.0,RoomService_8151.0,RoomService_8168.0,RoomService_8586.0,RoomService_9920.0
0,39,0,0,0,0,0,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,24,9,25,549,44,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,58,3576,0,6715,49,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,33,1283,371,3329,193,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,16,70,151,565,2,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [48]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y,
    test_size=0.2,
    random_state=SEED)
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')

X_train shape: (5284, 1111)
X_test shape:  (1322, 1111)


**Model Selection**

In this exercise we will be using **KNN** as our predictive model.

In [49]:
scaler = StandardScaler()
scaler.set_output(transform="pandas")
# set_output(transform="pandas") makes the scaler return a DataFrame (keeps the real column names and index)

# Fit ONLY on X_train --> avoids leaking any information from the test set into the scaling parameters
X_train_scaled_df = scaler.fit_transform(X_train)

# Then apply that same transformation to X_test
X_test_scaled_df = scaler.transform(X_test)

# BEFORE: original scale
print('BEFORE scaling (X_train):')
print(X_train.describe().loc[['mean', 'std']].round(4))
print()

# AFTER: standardized scale
print('AFTER scaling (X_train):')
print(X_train_scaled_df.describe().loc[['mean', 'std']].round(4))

BEFORE scaling (X_train):
          Age  FoodCourt  ShoppingMall        Spa     VRDeck  Transported  \
mean  28.8908   482.2347      179.9748   315.3403   306.1493       0.5021   
std   14.5046  1688.8443      590.1172  1136.2156  1139.2716       0.5000   

      HomePlanet_Europa  HomePlanet_Mars  CryoSleep_True  Cabin_B  ...  \
mean             0.2494           0.2120          0.3531   0.0897  ...   
std              0.4327           0.4087          0.4780   0.2858  ...   

      RoomService_6103.0  RoomService_6256.0  RoomService_6899.0  \
mean              0.0002              0.0002              0.0002   
std               0.0138              0.0138              0.0138   

      RoomService_7172.0  RoomService_7406.0  RoomService_8142.0  \
mean              0.0002              0.0002              0.0002   
std               0.0138              0.0138              0.0138   

      RoomService_8151.0  RoomService_8168.0  RoomService_8586.0  \
mean              0.0002              0.0

In [50]:
knn_model = KNeighborsClassifier(n_neighbors=5, weights='distance')
knn_model.fit(X_train_scaled_df, y_train)  # Fit on train

print('Model trained!')
# KNN does NOT learn coefficients (no coef_) --> it just memorizes the training points and their distances!!!

Model trained!


- Evaluate your model's performance. Comment it

In [53]:


# Evaluate the model on both Train and Test, to check for overfitting
y_pred_train = knn_model.predict(X_train_scaled_df)  # train
y_pred_test = knn_model.predict(X_test_scaled_df)    # test

accuracy_train = accuracy_score(y_train, y_pred_train)  # train, just to compare with the report below
print(f'Accuracy (Train): {accuracy_train * 100:.2f}%')
print()
print(classification_report(y_test, y_pred_test))

Accuracy (Train): 100.00%

              precision    recall  f1-score   support

       False       0.91      0.91      0.91       648
        True       0.92      0.91      0.91       674

    accuracy                           0.91      1322
   macro avg       0.91      0.91      0.91      1322
weighted avg       0.91      0.91      0.91      1322



In [ ]:
#It's oversampling - Accuracy 100%